## Loading Dataframe

In [ ]:
!pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 50.4 MB/s eta 0:00:00


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import seaborn as sns
from scipy import stats

In [ ]:
df_full_clean = pd.read_csv("maob_qsar_with_descriptors_clean.csv", sep=",")
df_full_clean.head()

,Smiles,pChEMBL Value,Molecular Weight,AlogP,#RO5 Violations,MaxAbsEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,svd_fp_40,svd_fp_41,svd_fp_42,svd_fp_43,svd_fp_44,svd_fp_45,svd_fp_46,svd_fp_47,svd_fp_48,svd_fp_49
0,Br.CCOC(=O)C(C(=O)N/N=C/c1ccc([N+](=O)[O-])cc1...,7.698970,488.32,2.29,0.0,12.693838,0.000000,-1.260586,0.137229,11.516129,...,-0.829413,-0.694527,-0.293248,-1.221602,0.004044,-0.971967,-0.325949,0.237666,1.306564,0.264758
1,Br.CCOc1ccc(-c2n[nH]cc2C2=NCCN2)cc1,1.230000,337.22,1.83,0.0,5.443938,0.000000,0.000000,0.900634,13.350000,...,0.202748,-0.312357,0.566727,-0.000873,-0.448249,0.027202,0.722492,0.337339,0.352410,-0.200908
2,Br.COc1ccc(C2=NCCN2)cc1Cn1ccnc1C,1.230000,351.25,1.60,0.0,5.457596,0.000000,0.000000,0.918740,13.333333,...,-0.059672,0.622384,0.310246,-0.135326,-0.111411,0.109594,0.777772,0.549119,-0.197255,-0.690708
3,Br.c1ccc(-n2cc(C3=NCCN3)c(-c3cccs3)n2)cc1,4.735182,375.30,2.95,0.0,4.773750,0.000000,0.000000,0.761197,13.363636,...,0.370238,0.504676,0.528244,-0.107077,-0.071850,0.570685,0.348111,0.098175,0.270774,-0.516726
4,Brc1ccc(-c2cc3ccccc3o2)cc1,6.468521,273.13,4.86,0.0,5.778639,0.910833,0.910833,0.616568,10.812500,...,0.262604,-0.090144,-0.610670,-0.014150,0.486537,-0.063679,-0.341234,-0.213476,0.084889,-0.230301


## 2.1 Model-Specific Feature Generation

Logarithmization of molecular weight

Physical meaning: the effect of weight on activity is often not linear, but decays.

In [ ]:
df_full_clean['Log_MolWt'] = np.log1p(df_full_clean['Molecular Weight'])

Lipophilicity Index

The interaction of LogP and weight is a classic "Drug-likeness" parameter.

In [ ]:
df_full_clean['LogP_Wt_Ratio'] = df_full_clean['AlogP'] / (df_full_clean['Molecular Weight'] + 1e-6)

## 2.2 Train–Test Split: Motivation & Proofs

https://www.datagrok.ai/help/datagrok/solutions/domains/chem/scripts/murcko-scaffolds

Using a scaffold-based split instead of random splitting allows evaluation of the model’s ability to extrapolate to unseen chemical classes. This is a critical test for overfitting, ensuring that the model learns general chemical patterns rather than memorizing specific molecular series.

In [ ]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit

# Getting groups of molecules
def get_scaffold_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            scaffold_mol = MurckoScaffold.GetScaffoldForMol(mol)
            return Chem.MolToSmiles(scaffold_mol)
        else:
            return "Invalid"
    except:
        return "Error"

df_full_clean['scaffold'] = df_full_clean['Smiles'].apply(get_scaffold_smiles)

# Target
y = df_full_clean['pChEMBL Value']

# Features matrix
cols_to_drop = ['Smiles', 'pChEMBL Value', 'scaffold']
X = df_full_clean.drop(columns=[c for c in cols_to_drop if c in df_full_clean.columns])

# Scaffold Split
groups = df_full_clean['scaffold']
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"\n--- (Scaffold Split) ---")
print(f"Train: {X_train.shape[0]} molecules")
print(f"Test:  {X_test.shape[0]} molecules")
print(f"Unique Scaffolds in Test: {groups.iloc[test_idx].nunique()}")


--- (Scaffold Split) ---
Train: 4660 molecules
Test:  1224 molecules
Unique Scaffolds in Test: 399


Scaffold Hopping refers to the ability of a model (or a chemist) to identify active molecules that belong to entirely new chemical scaffolds—structural cores not seen during training—while retaining the desired biological activity.


A standard random split can place very similar molecules (analogs) into both the training and test sets. As a result, the model may simply “memorize” these series, leading to artificially high $R^2$ values that overestimate its true predictive ability.

## 2.3 Leakage Detection

In [ ]:
train_scaffolds = set(groups.iloc[train_idx])
test_scaffolds = set(groups.iloc[test_idx])
overlap = train_scaffolds.intersection(test_scaffolds)
print(f"Intersection of scaffolds between Train and Test: {len(overlap)}")

Intersection of scaffolds between Train and Test: 0


A leakage test confirmed complete sample isolation: the scaffold overlap between Train and Test is 0. This guarantees the absence of identical structural backbones in both groups. The zero overlap coefficient confirms strict adherence to the Scaffold Split methodology, eliminating any inflated metrics due to the model "remembering" familiar chemical classes.

## 2.4 Feature Selection


*   Filter Method (Correlation Analysis)



We first applied a filter-based approach to reduce feature redundancy. Specifically, features with a Pearson correlation coefficient greater than 0.90 were identified and removed. Highly correlated features provide overlapping information, which can increase multicollinearity, add noise, and lead to overfitting.

* Scaffolds were added as features to support cross-validation and scaffold-based splitting.

* All other molecular descriptors and fingerprints were processed and prepared during the preprocessing stage, including correlation-based filtering, dimensionality reduction, and outlier handling.

## 2.5 Modelling Approach & Model Selection

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.6 MB/s eta 0:00:00


### Models and metrics

Were selected two models—Random Forest and CatBoost—and trained as baseline regressors to compare their initial performance before proceeding with full optimization.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold

cv = GroupKFold(n_splits=5)
groups_train = df_full_clean.iloc[train_idx]['scaffold']

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf_cv_scores = cross_val_score(rf, X_train, y_train, cv=cv, groups=groups_train, scoring='r2')
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
rf_test_r2 = r2_score(y_test, y_pred_rf)
rf_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

In [ ]:
cat = CatBoostRegressor(
    iterations=500, depth=6, learning_rate=0.1, verbose=0, random_state=42
)
cat_cv_scores = cross_val_score(cat, X_train, y_train, cv=cv, groups=groups_train, scoring='r2')
cat.fit(X_train, y_train)
y_pred_cat = cat.predict(X_test)
cat_test_r2 = r2_score(y_test, y_pred_cat)
cat_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_cat))

In [ ]:
print("=== Cross-Validation R² ===")
print(f"Random Forest: {rf_cv_scores.mean():.3f} ± {rf_cv_scores.std():.3f}")
print(f"CatBoost     : {cat_cv_scores.mean():.3f} ± {cat_cv_scores.std():.3f}")

print("\n=== Test Set Metrics ===")
print(f"Random Forest: R²={rf_test_r2:.3f}, RMSE={rf_test_rmse:.3f}")
print(f"CatBoost     : R²={cat_test_r2:.3f}, RMSE={cat_test_rmse:.3f}")

=== Cross-Validation R² ===
Random Forest: 0.520 ± 0.078
CatBoost     : 0.536 ± 0.085

=== Test Set Metrics ===
Random Forest: R²=0.561, RMSE=1.522
CatBoost     : R²=0.570, RMSE=1.507


CatBoost was chosen as the final model over Random Forest for several professional reasons. Although the performance difference seems small, CatBoost consistently achieved higher metrics, with cross-validation $R^2$ of 0.536 versus 0.520, test $R^2$ of 0.570 versus 0.561, and lower RMSE, which is meaningful in QSAR as it reflects better extraction of signal from complex chemical structures.

Its Ordered Boosting algorithm provides robustness against overfitting and ensures stable generalization across cross-validation and test sets. CatBoost also handles ordinal features, such as #RO5 Violations, natively, allowing the model to leverage this information more effectively than Random Forest.

As the t-SNE plot shows, the data contains clusters with sharply varying activity. CatBoost, using a gradient boosting algorithm on symmetric trees, copes with such nonlinear dependencies more effectively than Random Forest or linear models.

CatBoost offers flexible hyperparameters that can be fine-tuned via Optuna, enabling maximization of predictive performance in the final pipeline. Finally, CatBoost library natively supports the calculation of SHAP values, which will allow you to confirm the chemical adequacy of the model and ensure that the model does not "hallucinate" on random features.